# Name That LLM — Gaussian robusto + atribuição por prompt

Inferência offline com BGE-small, RoBERTa-base, DeBERTa-v3-base, Qwen3-0.6B e
StyleDistance. Os parâmetros ajustados e o TF-IDF portátil são arrays NumPy/JSON.
Grupos inferidos com menos de quatro textos recebem a média TF-IDF + quatro LR;
os demais recebem o Gaussian de cinco vistas. Não há quotas de classes.

O agrupamento e a inferência Gaussian recebem o conjunto de teste completo.
Somente os forwards dos encoders são divididos em lotes. Os campos utilizados
do teste são `id` e `text`. A execução valida hashes e falha claramente se um
componente estiver ausente ou inválido.

Dados privados: `juanseveriano/name-that-llm-encoders` e
`juanseveriano/name-that-llm-gaussiano`. Sem internet e sem instalação de pacotes.

Novidade: cada modelo escreve no máximo um texto por prompt. Os textos de um mesmo prompt são agrupados pelo
conteúdo (TF-IDF + BGE) e, em cada grupo, os rótulos são distintos (algoritmo húngaro sobre a mistura de
70% Gaussian e 30% média dos classificadores).


In [ ]:
import sys
import types

g = types.ModuleType('gaussiano_embutido')
sys.modules[g.__name__] = g
exec('"""Candidato local de Gaussian compartilhado por prompt, sem dependência de HF/torch.\n\nConfiguração congelada na campanha: cinco vistas, PCA residual até 128 dimensões,\ntemperatura 3, clusters mixed-content/BGE com average linkage e corte 0.60.\nTreino usa prompt_id; inferência seleciona exclusivamente id/text. Não há quotas,\nSinkhorn, repulsão de classes ou suposição de uma resposta de cada classe.\n"""\n\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.cluster.hierarchy import fcluster, linkage\nfrom scipy.spatial.distance import squareform\nfrom scipy.special import softmax\nfrom sklearn.covariance import ledoit_wolf_shrinkage\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.preprocessing import StandardScaler, normalize\nfrom sklearn.utils.extmath import randomized_svd\n\n\nVERSAO = 1\nSEMENTE = 20261103\nVISTAS = {\n    "bge-small-en-v1.5": {"backbone": "BAAI/bge-small-en-v1.5", "camadas": [2, 4, 6]},\n    "roberta-base": {"backbone": "FacebookAI/roberta-base", "camadas": [2, 4, 6, 8, 10, 12]},\n    "deberta-v3-base": {"backbone": "microsoft/deberta-v3-base", "camadas": [4, 8, 10, 11, 12]},\n    "Qwen3-0.6B": {"backbone": "Qwen/Qwen3-0.6B", "camadas": [0, 4, 6, 18, 20]},\n    "StyleDistance_L12": {"backbone": "StyleDistance/styledistance", "camadas": [12]},\n}\nCONFIG = {"dimensao_maxima": 128, "temperatura": 3., "limiar_grupo": .60,\n          "pca_iteracoes": 4, "two_way_iteracoes": 8, "cavity_iteracoes": 6,\n          "semente": SEMENTE, "lambda_minimo": 1e-4, "classe_unica_por_grupo": False,\n          "quota_classes": False, "repel": False, "sinkhorn": False}\n\n\n@dataclass\nclass Matrizes:\n    """IDs acompanham obrigatoriamente a ordem das linhas de todas as matrizes."""\n    ids: np.ndarray\n    vistas: dict[str, np.ndarray]\n    semantica: np.ndarray\n\n\n@dataclass\nclass Modelo:\n    classes: np.ndarray\n    treino_ids: np.ndarray\n    parametros: dict[str, dict[str, np.ndarray]]\n    metadados: dict\n\n\n@dataclass\nclass Resultado:\n    submissao: pd.DataFrame\n    probabilidades: np.ndarray\n    grupos: np.ndarray\n    diagnostico: dict\n\n\ndef checksum_ids(ids):\n    return hashlib.sha256("\\n".join(np.asarray(ids, dtype=str)).encode()).hexdigest()\n\n\ndef _sha(arquivo):\n    h = hashlib.sha256()\n    with Path(arquivo).open("rb") as f:\n        for bloco in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(bloco)\n    return h.hexdigest()\n\n\ndef _ids_textos(df):\n    if not {"id", "text"} <= set(df.columns):\n        raise ValueError("dados precisam de id e text")\n    ids = df["id"]\n    if ids.isna().any() or ids.duplicated().any() or not all(isinstance(i, str) and i for i in ids):\n        raise ValueError("ids precisam ser strings não vazias, únicas e completas")\n    if not all(isinstance(t, str) for t in df["text"]):\n        raise ValueError("text precisa conter strings, incluindo string vazia quando necessário")\n    return ids.to_numpy(dtype=str), df["text"].tolist()\n\n\ndef _matriz(a, linhas, nome):\n    a = np.asarray(a)\n    if a.ndim != 2 or a.shape[0] != linhas or a.shape[1] < 1 or a.dtype.kind not in "fiu":\n        raise ValueError(f"shape/dtype inválido em {nome}")\n    if not np.isfinite(a).all():\n        raise ValueError(f"valores não finitos em {nome}")\n    with np.errstate(over="ignore", invalid="ignore"):\n        convertido = np.asarray(a, dtype=np.float32)\n    if not np.isfinite(convertido).all():\n        raise ValueError(f"conversão float32 não finita em {nome}")\n    return convertido\n\n\ndef _validar_matrizes(ids, matrizes, modelo=None):\n    if not isinstance(matrizes, Matrizes):\n        raise TypeError("features precisam de Matrizes com IDs explícitos")\n    if not np.array_equal(np.asarray(matrizes.ids, dtype=str), ids):\n        raise ValueError("ordem/identidade dos IDs das matrizes difere dos dados")\n    if set(matrizes.vistas) != set(VISTAS):\n        raise ValueError("vistas ausentes ou inesperadas; configuração de cinco vistas é congelada")\n    vistas = {k: _matriz(matrizes.vistas[k], len(ids), k) for k in VISTAS}\n    semantica = _matriz(matrizes.semantica, len(ids), "semantica")\n    if modelo is not None:\n        if semantica.shape[1] != modelo.metadados["dimensao_semantica"]:\n            raise ValueError("dimensão semântica difere do modelo")\n        for k, x in vistas.items():\n            if x.shape[1] != len(modelo.parametros[k]["media"]):\n                raise ValueError(f"dimensão da vista {k} difere do modelo")\n    return vistas, semantica\n\n\ndef _centrar(X, grupos):\n    _, inverso, contagens = np.unique(grupos, return_inverse=True, return_counts=True)\n    medias = np.zeros((len(contagens), X.shape[1]), dtype=X.dtype)\n    np.add.at(medias, inverso, X)\n    medias /= contagens[:, None]\n    return X - medias[inverso]\n\n\ndef _padronizar(X, parametros):\n    # Mesma aritmética float32 in-place do StandardScaler.transform usado na pesquisa.\n    X = np.asarray(X, dtype=np.float32).copy()\n    X -= np.asarray(parametros["media"], dtype=np.float32)\n    X /= np.asarray(parametros["escala"], dtype=np.float32)\n    return X\n\n\ndef _ajustar_vista(X, y, grupos, classes):\n    escala = StandardScaler().fit(X)\n    Z = _padronizar(X, {"media": escala.mean_, "escala": escala.scale_})\n    residual = _centrar(Z, grupos)\n    d = min(CONFIG["dimensao_maxima"], len(X)-len(np.unique(grupos))-1, X.shape[1])\n    if d < 1:\n        raise ValueError("amostras insuficientes para PCA residual dentro dos prompts")\n    _, _, vt = randomized_svd(residual, n_components=d, n_iter=CONFIG["pca_iteracoes"], random_state=SEMENTE)\n    V = vt.T\n    Z = np.asarray(Z @ V, dtype=np.float64)\n    _, inv, contagem = np.unique(grupos, return_inverse=True, return_counts=True)\n    efeito = np.zeros((len(contagem), d))\n    mu = np.zeros((classes, d))\n    for _ in range(CONFIG["two_way_iteracoes"]):\n        R = Z - efeito[inv]\n        for c in range(classes):\n            mu[c] = R[y == c].mean(axis=0)\n        efeito[:] = 0\n        np.add.at(efeito, inv, Z - mu[y])\n        efeito /= contagem[:, None]\n        deslocamento = efeito.mean(axis=0)\n        efeito -= deslocamento\n        mu += deslocamento\n    E = Z - mu[y] - efeito[inv]\n    Se_raw = E.T @ E / max(len(X)-len(contagem)-classes+1, 1)\n    ae = float(ledoit_wolf_shrinkage(E, assume_centered=True))\n    Se = (1-ae)*Se_raw + ae*np.trace(Se_raw)/d*np.eye(d)\n    Se += np.eye(d)*max(float(np.trace(Se)/d*1e-7), 1e-9)\n    P = efeito - efeito.mean(axis=0)\n    Sp_raw = P.T @ P/max(len(P)-1, 1) - np.mean(1/contagem)*Se_raw\n    ap = float(ledoit_wolf_shrinkage(P, assume_centered=True))\n    tr = max(float(np.trace(Sp_raw)), 1e-8*float(np.trace(Se)))\n    Sp = (1-ap)*Sp_raw + ap*tr/d*np.eye(d)\n    Li = np.linalg.inv(np.linalg.cholesky(Se))\n    M = Li @ Sp @ Li.T\n    valores, Q = np.linalg.eigh((M+M.T)/2)\n    lam = np.maximum(valores, CONFIG["lambda_minimo"])\n    A = Q.T @ Li\n    return {"media": escala.mean_, "escala": escala.scale_, "V": V,\n            "A": A, "prototipos": mu @ A.T, "lambda": lam}, {"dimensao": d, "shrink_residual": ae, "shrink_prompt": ap}\n\n\ndef fit(treino: pd.DataFrame, matrizes: Matrizes) -> Modelo:\n    ids, _ = _ids_textos(treino)\n    if len(ids) == 0 or not {"model", "prompt_id"} <= set(treino.columns):\n        raise ValueError("treino precisa de linhas, model e prompt_id")\n    if not all(isinstance(v, str) and v for v in treino["model"]) or treino["prompt_id"].isna().any():\n        raise ValueError("rótulos/prompts inválidos")\n    classes = np.unique(treino["model"].to_numpy(dtype=str))\n    if len(classes) < 2:\n        raise ValueError("treino precisa de pelo menos duas classes")\n    y = np.searchsorted(classes, treino["model"].to_numpy(dtype=str))\n    grupos = treino["prompt_id"].to_numpy()\n    vistas, sem = _validar_matrizes(ids, matrizes)\n    parametros, infos = {}, {}\n    for k, x in vistas.items():\n        parametros[k], infos[k] = _ajustar_vista(x, y, grupos, len(classes))\n    metadados = {"versao": VERSAO, "config": dict(CONFIG), "vistas": json.loads(json.dumps(VISTAS)),\n                 "classes": classes.tolist(), "treino_linhas": len(ids), "treino_prompts": len(np.unique(grupos)),\n                 "treino_ids_sha256": checksum_ids(ids), "dimensao_semantica": sem.shape[1],\n                 "semantica": {"backbone": "BAAI/bge-small-en-v1.5", "camada": 12, "pooling": "mean tokens"},\n                 "ajuste": infos, "limites": ["grupos k=1/k=3 podem regredir", "probabilidades T3 subconfidentes", "ganho local não garante leaderboard"]}\n    modelo = Modelo(classes, ids, parametros, metadados)\n    _validar_modelo(modelo)\n    return modelo\n\n\ndef _grupos(textos, semantica):\n    if len(textos) < 2:\n        return np.arange(len(textos)), {"sem_vocabulario": not any(t.strip() for t in textos)}\n    sem_vocabulario = False\n    try:\n        X = TfidfVectorizer(stop_words="english", sublinear_tf=True, max_features=100_000, dtype=np.float32).fit_transform(textos)\n        distancia = np.clip(1-(X @ X.T).toarray(), 0, 1)\n    except ValueError as exc:\n        if "empty vocabulary" not in str(exc):\n            raise\n        sem_vocabulario = True\n        distancia = np.ones((len(textos), len(textos)))\n    E = np.asarray(semantica, dtype=np.float64)\n    E = normalize(E-E.mean(axis=0))\n    distancia = .5*(distancia+np.clip(1-E @ E.T, 0, 2))\n    np.fill_diagonal(distancia, 0)\n    grupos = fcluster(linkage(squareform(distancia, checks=False), method="average"), CONFIG["limiar_grupo"], criterion="distance")\n    return np.unique(grupos, return_inverse=True)[1], {"sem_vocabulario": sem_vocabulario}\n\n\ndef _probabilidades(X, parametros, grupos):\n    Z = _padronizar(X, parametros) @ parametros["V"]\n    U = np.asarray(Z, dtype=np.float64) @ parametros["A"].T\n    m, lam = parametros["prototipos"], parametros["lambda"]\n    R = U[:, None] - m[None]\n    p = softmax(-.5*np.sum(R**2/(1+lam), axis=2)/CONFIG["temperatura"], axis=1)\n    for g in np.unique(grupos):\n        ii = np.flatnonzero(grupos == g)\n        if len(ii) < 2:\n            continue\n        q = p[ii].copy()\n        v = lam/(1+(len(ii)-1)*lam)\n        for _ in range(CONFIG["cavity_iteracoes"]):\n            erros = U[ii]-q @ m\n            prompt = (erros.sum(axis=0)[None]-erros)*v\n            R = U[ii, None]-prompt[:, None]-m[None]\n            q = softmax(-.5*np.sum(R**2/(1+v), axis=2)/CONFIG["temperatura"], axis=1)\n        p[ii] = q\n    return p\n\n\ndef prever(modelo: Modelo, teste: pd.DataFrame, matrizes: Matrizes) -> Resultado:\n    """Somente id/text são lidos de teste; prompt_id/model extras são ignorados."""\n    _validar_modelo(modelo)\n    ids, textos = _ids_textos(teste)\n    vistas, sem = _validar_matrizes(ids, matrizes, modelo)\n    if len(ids) == 0:\n        return Resultado(pd.DataFrame({"id": ids, "model": np.array([], dtype=str)}), np.empty((0, len(modelo.classes))), np.empty(0, dtype=int), {"vazio": True})\n    grupos, diagnostico = _grupos(textos, sem)\n    p = np.mean([_probabilidades(vistas[k], modelo.parametros[k], grupos) for k in VISTAS], axis=0)\n    if p.shape != (len(ids), len(modelo.classes)) or not np.isfinite(p).all() or np.any(p < 0) or np.any(p.sum(axis=1) <= 0):\n        raise ValueError("probabilidades inválidas; nenhuma previsão constante de fallback será criada")\n    p /= p.sum(axis=1, keepdims=True)\n    saida = pd.DataFrame({"id": ids, "model": modelo.classes[p.argmax(axis=1)]})\n    if saida["id"].tolist() != ids.tolist() or not set(saida["model"]) <= set(modelo.classes):\n        raise ValueError("IDs/classes de saída inconsistentes")\n    _, inv, n = np.unique(grupos, return_inverse=True, return_counts=True)\n    diagnostico.update({"linhas": len(ids), "grupos": len(n), "fracao_em_grupos": float(np.mean(n[inv] > 1)),\n                       "classes_da_chamada_usadas": False, "prompt_id_da_chamada_usado": False})\n    return Resultado(saida, p, grupos, diagnostico)\n\n\ndef _validar_modelo(modelo):\n    m = modelo.metadados\n    if m.get("versao") != VERSAO or m.get("config") != CONFIG or m.get("vistas") != VISTAS:\n        raise ValueError("versão/configuração/vistas do modelo inválidas")\n    classes = np.asarray(modelo.classes)\n    if classes.ndim != 1 or classes.dtype.kind not in "US" or len(classes) < 2 or not np.array_equal(classes, np.unique(classes)) or any(not c for c in classes):\n        raise ValueError("classes precisam ser strings únicas na ordem canônica")\n    if classes.tolist() != m.get("classes"):\n        raise ValueError("ordem das classes difere dos metadados")\n    ids = np.asarray(modelo.treino_ids)\n    if ids.ndim != 1 or ids.dtype.kind not in "US" or len(ids) != m.get("treino_linhas") or len(np.unique(ids)) != len(ids) or checksum_ids(ids) != m.get("treino_ids_sha256"):\n        raise ValueError("identidade/ordem do treino inconsistente")\n    if set(modelo.parametros) != set(VISTAS):\n        raise ValueError("parâmetros de vistas ausentes/inesperados")\n    for k, p in modelo.parametros.items():\n        if set(p) != {"media", "escala", "V", "A", "prototipos", "lambda"}:\n            raise ValueError(f"parâmetros incompletos em {k}")\n        if any(np.asarray(v).dtype.kind not in "fiu" or not np.isfinite(v).all() for v in p.values()):\n            raise ValueError(f"parâmetros não finitos/não numéricos em {k}")\n        D = len(p["media"])\n        d = len(p["lambda"])\n        if p["media"].shape != (D,) or p["escala"].shape != (D,) or p["V"].shape != (D,d) or p["A"].shape != (d,d) or p["prototipos"].shape != (len(classes),d) or not 1 <= d <= CONFIG["dimensao_maxima"]:\n            raise ValueError(f"shapes de parâmetros inconsistentes em {k}")\n        if np.any(p["escala"] <= 0) or np.any(p["lambda"] <= 0):\n            raise ValueError(f"escala/covariância inválida em {k}")\n\n\ndef salvar(modelo: Modelo, destino):\n    _validar_modelo(modelo)\n    destino = Path(destino)\n    destino.mkdir(parents=True, exist_ok=True)\n    arrays = {"classes": np.asarray(modelo.classes, dtype=str), "treino_ids": np.asarray(modelo.treino_ids, dtype=str)}\n    for i, k in enumerate(VISTAS):\n        arrays.update({f"v{i}_{n}": np.asarray(v) for n, v in modelo.parametros[k].items()})\n    temporario = destino / "parametros.npz.tmp"\n    with temporario.open("wb") as f:\n        np.savez_compressed(f, **arrays)\n    temporario.replace(destino / "parametros.npz")\n    metadata = dict(modelo.metadados) | {"npz_sha256": _sha(destino / "parametros.npz"), "formato": "numpy arrays, allow_pickle=False"}\n    temporario = destino / "metadados.json.tmp"\n    temporario.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\\n")\n    temporario.replace(destino / "metadados.json")\n\n\ndef carregar(origem) -> Modelo:\n    origem = Path(origem)\n    metadata = json.loads((origem / "metadados.json").read_text())\n    if _sha(origem / "parametros.npz") != metadata.get("npz_sha256"):\n        raise ValueError("checksum NPZ inválido")\n    with np.load(origem / "parametros.npz", allow_pickle=False) as arq:\n        esperados={"classes","treino_ids"}|{f"v{i}_{n}" for i in range(len(VISTAS)) for n in ("media","escala","V","A","prototipos","lambda")}\n        if set(arq.files)!=esperados:\n            raise ValueError("campos NPZ ausentes/inesperados")\n        # Preservar C/F-order: mudar o layout altera a ordem de acumulação de BLAS\n        # no produto float32 e quebraria paridade estrita apesar dos mesmos valores.\n        parametros = {k: {n: arq[f"v{i}_{n}"].copy(order="K") for n in ("media", "escala", "V", "A", "prototipos", "lambda")} for i, k in enumerate(VISTAS)}\n        modelo = Modelo(arq["classes"].copy(), arq["treino_ids"].copy(), parametros, metadata)\n    _validar_modelo(modelo)\n    return modelo\n', g.__dict__)

robusto = types.ModuleType('gaussiano_robusto_embutido')
sys.modules[robusto.__name__] = robusto
robusto.g = g
exec('"""Política conservadora local, separada do Gaussian puro.\n\nGrupos INFERIDOS menores que quatro recebem mean5 (TF-IDF + quatro LR).\nOs demais recebem o Gaussian puro. Nenhuma exceção vira previsão constante.\nScalers/LR/TF-IDF são arrays portáteis: inferência não ajusta classificadores,\nnão usa pickle, labels de teste, torch, transformers ou quotas por classe.\n"""\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nfrom scipy.special import expit, softmax\nfrom sklearn.feature_extraction.text import CountVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler, normalize\n\n\nBASES=tuple(k for k in g.VISTAS if k!="StyleDistance_L12")\nCONFIG={"versao":1,"menor_grupo_gaussiano":4,"alpha_gaussiano_grupo_elegivel":1.,\n        "baseline":"mean5_tfidf_quatro_lr","C_embeddings":.05,"max_iter_embeddings":1000,\n        "sinkhorn":False,"quota_classes":False}\nTOKENS=r"(?u)\\b\\w+\\b|[^\\w\\s]"\nTFIDF_PARAMS={"palavras":{"token_pattern":TOKENS,"ngram_range":(1,2)},\n              "caracteres":{"analyzer":"char","ngram_range":(1,5)}}\n\n\n@dataclass\nclass Modelo:\n    gaussiano:g.Modelo\n    lineares:dict[str,dict[str,np.ndarray]]\n    vocabulario:dict[str,list[str]]\n    pesos_tfidf:dict[str,np.ndarray]\n    metadados:dict\n\n\ndef _sha(path):\n    h=hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)\n    return h.hexdigest()\n\n\ndef _linear(X,p):\n    X=np.asarray(X,dtype=np.float32).copy()\n    X-=p["media"].astype(np.float32)\n    X/=p["escala"].astype(np.float32)\n    scores=X@p["coef"].T+p["intercepto"]\n    if scores.shape[1]==1:\n        positive=expit(scores[:,0])\n        return np.column_stack((1-positive,positive))\n    return softmax(scores,axis=1)\n\n\ndef _tfidf(textos,modelo):\n    blocks=[]\n    for name in ("palavras","caracteres"):\n        counts=CountVectorizer(vocabulary=modelo.vocabulario[name],lowercase=False,**TFIDF_PARAMS[name]).transform(textos).astype(np.float64)\n        counts.data=1+np.log(counts.data)\n        blocks.append(normalize(counts@sp.diags(modelo.pesos_tfidf[f"idf_{name}"])))\n    X=sp.hstack(blocks).tocsr()\n    scores=X@modelo.pesos_tfidf["coef"].T+modelo.pesos_tfidf["intercepto"]\n    if scores.shape[1]==1:\n        positive=expit(scores[:,0])\n        return np.column_stack((1-positive,positive))\n    return softmax(scores,axis=1)\n\n\ndef fit(treino,matrizes,gaussiano,artefatos_tfidf):\n    """Ajusta somente os quatro classificadores fallback; Gaussian permanece fixo."""\n    g._validar_modelo(gaussiano)\n    ids,_=g._ids_textos(treino)\n    if not np.array_equal(ids,gaussiano.treino_ids):\n        raise ValueError("treino do fallback difere da ordem do Gaussian")\n    if not {"model"}<=set(treino.columns):raise ValueError("treino sem labels")\n    labels=treino.model.to_numpy(dtype=str)\n    if not np.array_equal(np.unique(labels),gaussiano.classes):raise ValueError("classes fallback/Gaussian diferentes")\n    vistas,_=g._validar_matrizes(ids,matrizes,gaussiano)\n    pasta=Path(artefatos_tfidf)\n    if json.loads((pasta/"rotulos.json").read_text())!=labels.tolist():\n        raise ValueError("ordem/labels de treino do TF-IDF incompatíveis")\n    vocabulario=json.loads((pasta/"tfidf_vocab.json").read_text())\n    with np.load(pasta/"tfidf.npz",allow_pickle=False) as f:\n        pesos={k:f[k].copy(order="K") for k in f.files}\n    lineares={}\n    for name in BASES:\n        pipe=make_pipeline(StandardScaler(),LogisticRegression(C=.05,max_iter=1000)).fit(vistas[name],labels)\n        scaler,clf=pipe.steps[0][1],pipe.steps[1][1]\n        lineares[name]={"media":scaler.mean_.copy(),"escala":scaler.scale_.copy(),\n                        "coef":clf.coef_.copy(order="K"),"intercepto":clf.intercept_.copy(),"classes":clf.classes_.astype(str)}\n        if not np.array_equal(lineares[name]["classes"],gaussiano.classes):raise ValueError("classificador fora da ordem canônica")\n    metadata={"config":dict(CONFIG),"classes":gaussiano.classes.tolist(),"treino_ids_sha256":g.checksum_ids(ids),\n              "treino_linhas":len(ids),"vistas_baseline":json.loads(json.dumps({k:g.VISTAS[k] for k in BASES})),\n              "tfidf_origem_sha256":_sha(pasta/"tfidf.npz"),"tfidf_vocab_origem_sha256":_sha(pasta/"tfidf_vocab.json"),\n              "regra_origem":"hipótese pós-diagnóstico testada no split20261105, sem retuning nesse split",\n              "proibido":"labels/prompt_id de teste, quotas de classe, fallback por exceção"}\n    modelo=Modelo(gaussiano,lineares,vocabulario,pesos,metadata)\n    _validar(modelo)\n    return modelo\n\n\ndef _validar(modelo):\n    g._validar_modelo(modelo.gaussiano)\n    meta=modelo.metadados\n    classes=modelo.gaussiano.classes\n    if meta.get("config")!=CONFIG or meta.get("classes")!=classes.tolist():raise ValueError("config/classes robustas incompatíveis")\n    if meta.get("treino_ids_sha256")!=g.checksum_ids(modelo.gaussiano.treino_ids) or meta.get("treino_linhas")!=len(modelo.gaussiano.treino_ids):\n        raise ValueError("checksum/ordem de treino fallback incompatível")\n    if set(modelo.lineares)!=set(BASES) or meta.get("vistas_baseline")!={k:g.VISTAS[k] for k in BASES}:raise ValueError("vistas fallback incompatíveis")\n    C=len(classes)\n    for name,p in modelo.lineares.items():\n        if set(p)!={"media","escala","coef","intercepto","classes"}:raise ValueError("campos LR inválidos")\n        if not np.array_equal(p["classes"],classes):raise ValueError("ordem das classes LR incompatível")\n        for k in ("media","escala","coef","intercepto"):\n            if np.asarray(p[k]).dtype.kind not in "fiu" or not np.isfinite(p[k]).all():raise ValueError("parâmetros LR não finitos")\n        D=len(p["media"]);rows=1 if C==2 else C\n        if p["media"].shape!=(D,) or p["escala"].shape!=(D,) or p["coef"].shape!=(rows,D) or p["intercepto"].shape!=(rows,):raise ValueError("shape LR inválido")\n        if D!=len(modelo.gaussiano.parametros[name]["media"]) or np.any(p["escala"]<=0):raise ValueError("dimensão/escala LR inválida")\n    if set(modelo.vocabulario)!={"palavras","caracteres"}:raise ValueError("vocabulários TF-IDF incompatíveis")\n    if set(modelo.pesos_tfidf)!={"idf_palavras","idf_caracteres","coef","intercepto"}:raise ValueError("pesos TF-IDF incompatíveis")\n    for name,terms in modelo.vocabulario.items():\n        if not isinstance(terms,list) or not terms or any(not isinstance(s,str) for s in terms) or len(set(terms))!=len(terms):raise ValueError("vocabulário TF-IDF inválido")\n        idf=modelo.pesos_tfidf[f"idf_{name}"]\n        if idf.shape!=(len(terms),) or np.any(idf<=0):raise ValueError("shape/idf TF-IDF inválido")\n    if any(np.asarray(v).dtype.kind not in "fiu" or not np.isfinite(v).all() for v in modelo.pesos_tfidf.values()):raise ValueError("pesos TF-IDF não finitos")\n    rows=1 if C==2 else C\n    if modelo.pesos_tfidf["coef"].shape!=(rows,sum(map(len,modelo.vocabulario.values()))) or modelo.pesos_tfidf["intercepto"].shape!=(rows,):raise ValueError("classes/shape TF-IDF incompatíveis")\n\n\ndef prever_baseline(modelo,teste,matrizes):\n    """Somente quatro vistas são lidas; StyleDistance/semântica não participam."""\n    _validar(modelo)\n    ids,textos=g._ids_textos(teste)\n    if not isinstance(matrizes,g.Matrizes) or not np.array_equal(ids,np.asarray(matrizes.ids,dtype=str)):raise ValueError("IDs do baseline fora de ordem")\n    if not set(BASES)<=set(matrizes.vistas):raise ValueError("vista baseline ausente")\n    if not len(ids):return np.empty((0,len(modelo.gaussiano.classes)))\n    parts=[_tfidf(textos,modelo)]\n    for name in BASES:\n        X=g._matriz(matrizes.vistas[name],len(ids),name)\n        if X.shape[1]!=len(modelo.lineares[name]["media"]):raise ValueError("shape de vista baseline inválido")\n        parts.append(_linear(X,modelo.lineares[name]))\n    p=np.mean(parts,axis=0)\n    if p.shape!=(len(ids),len(modelo.gaussiano.classes)) or not np.isfinite(p).all() or np.any(p<0) or np.any(p.sum(axis=1)<=0):raise ValueError("probabilidades baseline inválidas")\n    return p/p.sum(axis=1,keepdims=True)\n\n\ndef prever(modelo,teste,matrizes):\n    """Pool inteiro após extrair todas as features; batching somente nos encoders.\n\n    Roteamento explícito por tamanho inferido, sem capturar erros de modelos.\n    Particionar este método em chunks muda os grupos e as previsões.\n    """\n    _validar(modelo)\n    gaussian=g.prever(modelo.gaussiano,teste,matrizes)\n    baseline=prever_baseline(modelo,teste,matrizes)\n    if not len(baseline):return gaussian\n    _,inv,counts=np.unique(gaussian.grupos,return_inverse=True,return_counts=True)\n    mask=counts[inv]>=CONFIG["menor_grupo_gaussiano"]\n    p=baseline.copy();p[mask]=gaussian.probabilidades[mask]\n    if not np.isfinite(p).all():raise ValueError("roteamento gerou probabilidades não finitas")\n    ids,_=g._ids_textos(teste)\n    submission=pd.DataFrame({"id":ids,"model":modelo.gaussiano.classes[p.argmax(axis=1)]})\n    diagnostic=dict(gaussian.diagnostico)|{"politica":"grupo inferido <4 mean5; >=4 Gaussian", "linhas_gaussian":int(mask.sum()),\n                                            "linhas_mean5":int((~mask).sum()),"falhas_convertidas_em_fallback":False}\n    return g.Resultado(submission,p,gaussian.grupos,diagnostic)\n\n\ndef salvar(modelo,destino):\n    _validar(modelo)\n    destino=Path(destino);destino.mkdir(parents=True,exist_ok=True)\n    g.salvar(modelo.gaussiano,destino/"gaussiano")\n    arrays={"classes":modelo.gaussiano.classes}\n    for i,name in enumerate(BASES):arrays.update({f"b{i}_{k}":v for k,v in modelo.lineares[name].items()})\n    arrays.update({f"tfidf_{k}":v for k,v in modelo.pesos_tfidf.items()})\n    tmp=destino/"fallback.npz.tmp"\n    with tmp.open("wb") as f:np.savez_compressed(f,**arrays)\n    tmp.replace(destino/"fallback.npz")\n    tmp=destino/"tfidf_vocab.json.tmp";tmp.write_text(json.dumps(modelo.vocabulario,ensure_ascii=False));tmp.replace(destino/"tfidf_vocab.json")\n    metadata=dict(modelo.metadados)|{"fallback_npz_sha256":_sha(destino/"fallback.npz"),"vocab_json_sha256":_sha(destino/"tfidf_vocab.json"),\n                                     "gaussiano_npz_sha256":_sha(destino/"gaussiano/parametros.npz"),"formato":"allow_pickle=False"}\n    tmp=destino/"metadados.json.tmp";tmp.write_text(json.dumps(metadata,ensure_ascii=False,indent=2)+"\\n");tmp.replace(destino/"metadados.json")\n\n\ndef carregar(origem):\n    origem=Path(origem);metadata=json.loads((origem/"metadados.json").read_text())\n    for filename,key in (("fallback.npz","fallback_npz_sha256"),("tfidf_vocab.json","vocab_json_sha256"),("gaussiano/parametros.npz","gaussiano_npz_sha256")):\n        if _sha(origem/filename)!=metadata.get(key):raise ValueError(f"checksum inválido: {filename}")\n    gaussian=g.carregar(origem/"gaussiano")\n    with np.load(origem/"fallback.npz",allow_pickle=False) as f:\n        expected={"classes"}|{f"b{i}_{k}" for i in range(len(BASES)) for k in ("media","escala","coef","intercepto","classes")}|{f"tfidf_{k}" for k in ("idf_palavras","idf_caracteres","coef","intercepto")}\n        if set(f.files)!=expected:raise ValueError("campos fallback NPZ inválidos")\n        if not np.array_equal(f["classes"],gaussian.classes):raise ValueError("classes NPZ/Gaussian incompatíveis")\n        lineares={name:{k:f[f"b{i}_{k}"].copy(order="K") for k in ("media","escala","coef","intercepto","classes")} for i,name in enumerate(BASES)}\n        weights={k:f[f"tfidf_{k}"].copy(order="K") for k in ("idf_palavras","idf_caracteres","coef","intercepto")}\n    model=Modelo(gaussian,lineares,json.loads((origem/"tfidf_vocab.json").read_text()),weights,metadata)\n    _validar(model)\n    return model\n', robusto.__dict__)

PACOTE = {'versao': 1, 'modelos': {'bge-small-en-v1.5': {'backbone': 'BAAI/bge-small-en-v1.5', 'camadas': [2, 4, 6], 'hidden_size': 384, 'arquivos': {'config.json': {'bytes': 743, 'sha256': '094f8e891b932f2000c92cfc663bac4c62069f5d8af5b5278c4306aef3084750'}, 'model.safetensors': {'bytes': 133466304, 'sha256': '3c9f31665447c8911517620762200d2245a2518d6e7208acc78cd9db317e21ad'}, 'special_tokens_map.json': {'bytes': 125, 'sha256': 'b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3'}, 'tokenizer.json': {'bytes': 711396, 'sha256': 'd241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66'}, 'tokenizer_config.json': {'bytes': 366, 'sha256': '9261e7d79b44c8195c1cada2b453e55b00aeb81e907a6664974b4d7776172ab3'}, 'vocab.txt': {'bytes': 231508, 'sha256': '07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3'}}}, 'roberta-base': {'backbone': 'FacebookAI/roberta-base', 'camadas': [2, 4, 6, 8, 10, 12], 'hidden_size': 768, 'arquivos': {'config.json': {'bytes': 481, 'sha256': 'ef0185e2aae6e06c5f105a285006952c340e20c7dbf43c86ec82601b13fc45e9'}, 'merges.txt': {'bytes': 456318, 'sha256': '1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5'}, 'model.safetensors': {'bytes': 498818054, 'sha256': '5bde1d28afb363d0103324efeb5afc8b2b397fe5e04beabb9b1ef355255ade81'}, 'tokenizer.json': {'bytes': 1355863, 'sha256': '847bbeab6174d66a88898f729d52fa8d355fafe1bea101cf960dd404581df70e'}, 'tokenizer_config.json': {'bytes': 25, 'sha256': '994f46754c5bf4014f1aa92d34b1374319c3a6b3f702105cd5b742beaecd18ce'}, 'vocab.json': {'bytes': 898823, 'sha256': '9e7f63c2d15d666b52e21d250d2e513b87c9b713cfa6987a82ed89e5e6e50655'}}}, 'deberta-v3-base': {'backbone': 'microsoft/deberta-v3-base', 'camadas': [4, 8, 10, 11, 12], 'hidden_size': 768, 'arquivos': {'config.json': {'bytes': 579, 'sha256': '649f6a1ec33c6bdd9a6486d5c66019d461139e54957073eafe9bbc2d34c75b0b'}, 'model.safetensors': {'bytes': 371101258, 'sha256': '57cbd0cad054ba5be8d4c6965b836e132f029edbbe3ed9c5bc9ef4fe1c40c34e'}, 'spm.model': {'bytes': 2464616, 'sha256': 'c679fbf93643d19aab7ee10c0b99e460bdbc02fedf34b92b05af343b4af586fd'}, 'tokenizer.json': {'bytes': 8339876, 'sha256': 'e9ddd853a4a31023721c993458663f2019e3154a05dacf0d2b3c217e53d9bdc8'}, 'tokenizer_config.json': {'bytes': 466, 'sha256': 'e546fb51d863e730c4134398f701b596eb43a13d6494e88cc8301bef561d4adb'}}}, 'Qwen3-0.6B': {'backbone': 'Qwen/Qwen3-0.6B', 'camadas': [0, 4, 6, 18, 20], 'hidden_size': 1024, 'arquivos': {'config.json': {'bytes': 726, 'sha256': '660db3b73d788119c04535e48cf9be5f55bc3100841a718637ae695b442f27dd'}, 'merges.txt': {'bytes': 1671853, 'sha256': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5'}, 'model.safetensors': {'bytes': 1503300328, 'sha256': 'f47f71177f32bcd101b7573ec9171e6a57f4f4d31148d38e382306f42996874b'}, 'tokenizer.json': {'bytes': 11422654, 'sha256': 'aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4'}, 'tokenizer_config.json': {'bytes': 9732, 'sha256': 'd5d09f07b48c3086c508b30d1c9114bd1189145b74e982a265350c923acd8101'}, 'vocab.json': {'bytes': 2776833, 'sha256': 'ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910'}}}, 'StyleDistance_L12': {'backbone': 'StyleDistance/styledistance', 'camadas': [12], 'hidden_size': 768, 'arquivos': {'config.json': {'bytes': 678, 'sha256': 'b7bcc672f41211775356b1bb7b614453b91edc8425f58eb4ba2c71d6d8b1875a'}, 'merges.txt': {'bytes': 456318, 'sha256': '1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5'}, 'model.safetensors': {'bytes': 249313784, 'sha256': '6cd0908217a8110f068a1502f7e7157303bffbf965427f4a0b88a48b1b6544b1'}, 'special_tokens_map.json': {'bytes': 280, 'sha256': '06e405a36dfe4b9604f484f6a1e619af1a7f7d09e34a8555eb0b77b66318067f'}, 'tokenizer.json': {'bytes': 2108643, 'sha256': '6e6a06a662f8bce9823abd4c2cda07dcb7e07a84153a2ed190b8d901516b3162'}, 'tokenizer_config.json': {'bytes': 1215, 'sha256': '56f306b23b9233c94016f1b0b8aea3bc120676b378f70933908c9e62ab2daa36'}, 'vocab.json': {'bytes': 798293, 'sha256': 'ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e'}}}}, 'artefatos': {'fallback.npz': {'bytes': 22490875, 'sha256': 'f4ba1576f562fe8f0ccffd3d1effb8693ce36ce647e3b7348dbdfbe8beffd038'}, 'gaussiano/metadados.json': {'bytes': 3822, 'sha256': '0021c7baba6432a1725d9bec7a30ebfc92b3a6cfccb20a2cbf63da3e55df1af9'}, 'gaussiano/parametros.npz': {'bytes': 8325510, 'sha256': 'dd90029f1cbfe588e88cb2d2d9a21929102a94a2c5b6f832aba40e293e7c021b'}, 'metadados.json': {'bytes': 2109, 'sha256': '86ada923b5aff20dc326990faf43ad66a97e78071c0d36ec1f6a23edea9f18b7'}, 'tfidf_vocab.json': {'bytes': 1727694, 'sha256': 'a2fd1c5b72865eb68bf9a14806fe891ad3458209b65b54a738c26c1d09b3467d'}}, 'dataset_manifesto': {'sha256': '93e48c1213e28f3e745815f8334e42fc30fa1597dc340ad2b0c487484d3e1827', 'bytes': 7206, 'candidate': 'gaussian5_robust', 'format_version': 1}, 'representacao': {'max_length': 512, 'dtype_modelo': 'float32', 'dtype_features': 'float16', 'pooling': 'hidden_states masked mean; todos tokens válidos incluindo especiais; sem L2', 'semantica': 'bge-small-en-v1.5 hidden_states[12]', 'tokens_especiais': 'padrão do tokenizer', 'eos_manual': False, 'camadas_truncadas': False, 'autocast': False}, 'fontes_sha256': {'gaussiano.py': '225c05c31c0ac8822f7561e4888073c44a9f08f4b8bbfe19f4599a511048f172', 'gaussiano_robusto.py': 'bf55a5aa2cb8c5eecb765e6cdc0193927497076697a635db2cb57bc6e84e7bd0'}}

atribuicao = types.ModuleType('atribuicao_embutida')
sys.modules[atribuicao.__name__] = atribuicao
exec('"""Cada modelo escreve no máximo um texto por prompt, então os rótulos de um grupo de textos do mesmo prompt\nsão todos diferentes. Infere os grupos pelo texto (sem prompt_id) e escolhe, em cada um, a atribuição de\nrótulos distintos que maximiza a soma dos log-probabilidades (algoritmo húngaro).\n\nSó usa numpy, scipy e sklearn: vira parte do notebook do Kaggle.\n"""\n\nimport numpy as np\nfrom scipy.cluster.hierarchy import fcluster, linkage\nfrom scipy.optimize import linear_sum_assignment\nfrom scipy.spatial.distance import squareform\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.preprocessing import normalize\n\nCORTE = 0.8  # distância máxima de ligação média dentro de um grupo\nTETO = 18  # um texto por modelo: grupo maior que isso mistura prompts\n\n\ndef distancias(textos: list[str], semantica: np.ndarray) -> np.ndarray:\n    """Média da distância de cosseno em TF-IDF de conteúdo (sem stopwords) e em embeddings semânticos."""\n    try:\n        T = TfidfVectorizer(stop_words="english", sublinear_tf=True, max_features=100_000, dtype=np.float32).fit_transform(textos)\n        sim_t = (T @ T.T).toarray()\n    except ValueError:  # vocabulário vazio\n        sim_t = np.zeros((len(textos), len(textos)), dtype=np.float32)\n    E = np.asarray(semantica, dtype=np.float32)\n    E = normalize(E - E.mean(axis=0))\n    D = np.clip(1 - 0.5 * (sim_t + E @ E.T), 0, 2).astype(np.float64)\n    np.fill_diagonal(D, 0)\n    return D\n\n\ndef agrupar(D: np.ndarray, corte: float = CORTE, teto: int = TETO) -> np.ndarray:\n    """Corte da árvore de ligação média; grupo com mais de `teto` textos é recortado com cortes menores."""\n    rotulos = np.zeros(len(D), dtype=int)\n    if len(D) < 2:\n        return rotulos\n    proximo = 0\n    pilha = [(np.arange(len(D)), corte)]\n    while pilha:\n        idx, c = pilha.pop()\n        if len(idx) <= 1:\n            rotulos[idx], proximo = proximo, proximo + 1\n            continue\n        g = fcluster(linkage(squareform(D[np.ix_(idx, idx)], checks=False), method="average"), c, criterion="distance")\n        for k in np.unique(g):\n            membros = idx[g == k]\n            if len(membros) > teto and c > 0.2:\n                pilha.append((membros, c - 0.03))\n            else:\n                rotulos[membros], proximo = proximo, proximo + 1\n    return rotulos\n\n\ndef atribuir(logp: np.ndarray, grupos: np.ndarray) -> np.ndarray:\n    """Índice de classe de cada texto: húngaro por grupo; grupo de um texto ou com mais classes que texto sobra no argmax."""\n    pred = logp.argmax(axis=1)\n    for g in np.unique(grupos):\n        idx = np.where(grupos == g)[0]\n        if 1 < len(idx) <= logp.shape[1]:\n            linhas, colunas = linear_sum_assignment(-logp[idx])\n            pred[idx[linhas]] = colunas\n    return pred\n\n\ndef refinar(base, gauss, textos, semantica, grupos_gauss, menor_grupo: int = 4, peso_gauss: float = 0.7):\n    """Mistura os log-probabilidades (base e Gaussian; grupo pequeno do Gaussian só usa a base) e aplica o\n    húngaro nos grupos inferidos. Devolve (log-probabilidades misturados, índice de classe por texto)."""\n    L = np.log(np.clip(base, 1e-9, None))\n    LG = np.log(np.clip(gauss, 1e-9, None))\n    _, inv, contagem = np.unique(grupos_gauss, return_inverse=True, return_counts=True)\n    usa = (contagem[inv] >= menor_grupo)[:, None]\n    L = np.where(usa, (1 - peso_gauss) * L + peso_gauss * LG, L)\n    return L, atribuir(L, agrupar(distancias(textos, semantica)))\n', atribuicao.__dict__)


In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import time
import traceback
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer


def sha_arquivo(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for bloco in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def salvar_json(path, valor):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(valor, ensure_ascii=False, indent=2) + "\n")
    tmp.replace(path)


def conferir_arquivos(pasta, arquivos):
    for nome, esperado in arquivos.items():
        p = pasta / nome
        if not p.is_file():
            raise FileNotFoundError(f"Arquivo obrigatório ausente: {p}")
        if p.stat().st_size != esperado["bytes"] or sha_arquivo(p) != esperado["sha256"]:
            raise ValueError(f"SHA256/tamanho inválido: {p}")


def achar_teste(entrada):
    candidatos = []
    for p in sorted(entrada.rglob("test.csv")):
        colunas = pd.read_csv(p, nrows=0).columns
        if {"id", "text"} <= set(colunas):
            candidatos.append(p)
    if len(candidatos) != 1:
        raise ValueError(f"Esperado um test.csv com id/text; encontrados {len(candidatos)}: {candidatos}")
    return candidatos[0]


def achar_artefatos(entrada):
    esperado = PACOTE["artefatos"]["metadados.json"]["sha256"]
    candidatos = [p.parent for p in sorted(entrada.rglob("metadados.json"))
                  if p.parent.joinpath("fallback.npz").is_file() and sha_arquivo(p) == esperado]
    if len(candidatos) != 1:
        raise ValueError(f"Esperado um pacote robusto compatível; encontrados {len(candidatos)}")
    conferir_arquivos(candidatos[0], PACOTE["artefatos"])
    raiz_dataset = candidatos[0].parent
    conferir_arquivos(raiz_dataset, {"manifesto.json": PACOTE["dataset_manifesto"]})
    manifesto = json.loads((raiz_dataset / "manifesto.json").read_text())
    if manifesto.get("format_version") != 1 or manifesto.get("candidate") != "gaussian5_robust":
        raise ValueError("Schema/candidato do manifesto do dataset incompatível")
    for rel in manifesto["files"]:
        if Path(rel).is_absolute() or ".." in Path(rel).parts:
            raise ValueError("Path relativo inválido no manifesto do dataset")
    conferir_arquivos(raiz_dataset, manifesto["files"])
    return candidatos[0]


def achar_modelo(entrada, nome):
    spec = PACOTE["modelos"][nome]
    esperado = spec["arquivos"]["config.json"]["sha256"]
    candidatos = []
    for p in sorted(entrada.rglob("config.json")):
        if p.stat().st_size == spec["arquivos"]["config.json"]["bytes"] and sha_arquivo(p) == esperado:
            # RoBERTa e StyleDistance podem compartilhar a arquitetura, mas não os pesos.
            if all((p.parent / n).is_file() and (p.parent / n).stat().st_size == s["bytes"]
                   for n, s in spec["arquivos"].items()):
                candidatos.append(p.parent)
    validos = []
    for pasta in candidatos:
        try:
            conferir_arquivos(pasta, spec["arquivos"])
        except ValueError:
            continue
        validos.append(pasta)
    if len(validos) != 1:
        raise ValueError(f"Esperado um encoder {nome} com SHA256 compatível; encontrados {len(validos)}")
    return validos[0]


def extrair_vista(caminho, textos, camadas, *, dispositivo, lote=16, semantica=False):
    inicio = time.monotonic()
    tok = AutoTokenizer.from_pretrained(caminho, local_files_only=True, trust_remote_code=False)
    enc = AutoModel.from_pretrained(caminho, local_files_only=True, trust_remote_code=False,
                                    torch_dtype=torch.float32).float().to(dispositivo).eval()
    if any(p.dtype != torch.float32 for p in enc.parameters() if p.is_floating_point()):
        raise ValueError("Encoder precisa estar em float32")
    ordem = np.argsort([len(t) for t in textos])
    camadas_necessarias = sorted(set(camadas) | ({12} if semantica else set()))
    dim = enc.config.hidden_size
    features = np.empty((len(textos), len(camadas) * dim), dtype=np.float16)
    sem = np.empty((len(textos), dim), dtype=np.float16) if semantica else None
    truncados = 0
    comprimentos = tok(list(textos), add_special_tokens=True, truncation=False)["input_ids"]
    truncados = sum(len(x) > 512 for x in comprimentos)
    del comprimentos
    with torch.inference_mode():
        for inicio_lote in range(0, len(textos), lote):
            idx = ordem[inicio_lote:inicio_lote + lote]
            b = tok([textos[j] for j in idx], padding=True, truncation=True,
                    max_length=512, return_tensors="pt").to(dispositivo)
            states = enc(**b, output_hidden_states=True).hidden_states
            # Mesma média mascarada dos caches: inclui todos tokens especiais reais.
            hidden = torch.stack([states[c] for c in camadas_necessarias], dim=1)
            mask = b["attention_mask"][:, None, :, None].to(hidden.dtype)
            means = ((hidden * mask).sum(2) / mask.sum(2)).float().cpu().numpy().astype(np.float16)
            features[idx] = np.concatenate([means[:, camadas_necessarias.index(c)] for c in camadas], axis=1)
            if semantica:
                sem[idx] = means[:, camadas_necessarias.index(12)]
            del b, states, hidden, mask, means
    if not np.isfinite(features).all() or (sem is not None and not np.isfinite(sem).all()):
        raise ValueError(f"Features não finitas: {caminho}")
    diagnostico = {"caminho": str(caminho), "camadas": list(camadas), "shape": list(features.shape),
                   "dtype_modelo": "float32", "dtype_features": str(features.dtype), "max_length": 512,
                   "textos_truncados": truncados, "lote": lote, "segundos": time.monotonic() - inicio,
                   "features_sha256": hashlib.sha256(features.tobytes()).hexdigest(),
                   "semantica_sha256": hashlib.sha256(sem.tobytes()).hexdigest() if sem is not None else None}
    del enc, tok
    gc.collect()
    if dispositivo == "cuda":
        torch.cuda.empty_cache()
    elif dispositivo == "mps":
        torch.mps.empty_cache()
    return features, sem, diagnostico


def proibido_ajustar(*args, **kwargs):
    raise RuntimeError("Ajuste de classificadores proibido na inferência")


def executar(entrada=Path("/kaggle/input"), saida=Path("/kaggle/working"), dispositivo=None, lote=16):
    entrada, saida = Path(entrada), Path(saida)
    saida.mkdir(parents=True, exist_ok=True)
    if lote < 1:
        raise ValueError("Lote precisa ser positivo")
    if dispositivo is None:
        dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    if dispositivo == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
    inicio = time.monotonic()
    versoes = {n: importlib.metadata.version(n) for n in
               ("numpy", "pandas", "scipy", "scikit-learn", "torch", "transformers", "pyarrow")}
    try:
        versoes["sentencepiece"] = importlib.metadata.version("sentencepiece")
    except importlib.metadata.PackageNotFoundError:
        # Os tokenizers rápidos exportados usam tokenizer.json sem essa dependência.
        versoes["sentencepiece"] = "ausente (tokenizer.json rápido)"
    log = {"versoes": versoes, "python": platform.python_version(), "dispositivo": dispositivo,
           "pacote": PACOTE, "componentes": {}, "offline": True, "refit_classificadores": False,
           "batching": "somente encoders; Gaussian e agrupamento recebem pool inteiro"}
    salvar_json(saida / "diagnostico.json", log)
    print("Ambiente:", json.dumps({"versoes": versoes, "dispositivo": dispositivo}), flush=True)
    teste_path = achar_teste(entrada)
    teste = pd.read_csv(teste_path, usecols=["id", "text"], dtype={"id": str, "text": str}, keep_default_na=False)
    ids, textos = g._ids_textos(teste)
    if not len(ids):
        raise ValueError("test.csv sem linhas")
    log.update({"test_path": str(teste_path), "test_sha256": sha_arquivo(teste_path), "linhas": len(ids),
                "ids_sha256": g.checksum_ids(ids), "ids_completos": ids.tolist()})
    pasta_artefatos = achar_artefatos(entrada)
    modelo = robusto.carregar(pasta_artefatos)
    if len(modelo.gaussiano.classes) != 18:
        raise ValueError("Esperadas as 18 classes do candidato")
    # A presença das funções de treino na fonte embutida não permite usá-las aqui.
    g.fit = g._ajustar_vista = robusto.fit = proibido_ajustar
    vistas, semantica = {}, None
    for nome, spec in g.VISTAS.items():
        print("Encoder:", nome, "camadas", spec["camadas"], flush=True)
        caminho = achar_modelo(entrada, nome)
        X, sem, info = extrair_vista(caminho, textos, spec["camadas"], dispositivo=dispositivo,
                                     lote=lote, semantica=(nome == "bge-small-en-v1.5"))
        esperado = len(modelo.gaussiano.parametros[nome]["media"])
        if X.shape != (len(ids), esperado) or X.dtype != np.float16 or not np.isfinite(X).all():
            raise ValueError(f"Contrato de features inválido: {nome}")
        vistas[nome] = X
        if sem is not None:
            semantica = sem
        log["componentes"][nome] = info
        salvar_json(saida / "diagnostico.json", log)
        print("Encoder pronto:", nome, json.dumps(info), flush=True)
    if semantica is None or semantica.shape != (len(ids), modelo.gaussiano.metadados["dimensao_semantica"]):
        raise ValueError("Semântica BGE12 ausente/incompatível")
    matrizes = g.Matrizes(ids, vistas, semantica)
    # Uma chamada, sobre TODAS as linhas; nenhuma partição Gaussian em minibatches.
    # Os mesmos grupos servem ao Gaussian (remoção do tema) e à atribuição: corte 0,8 com teto de 18 textos.
    g._grupos = lambda txt, sem: (atribuicao.agrupar(atribuicao.distancias(txt, sem)), {"sem_vocabulario": False})
    previsao = robusto.prever(modelo, teste, matrizes)
    base = robusto.prever_baseline(modelo, teste, matrizes)
    gauss = g.prever(modelo.gaussiano, teste, matrizes)
    L, rotulos = atribuicao.refinar(base, gauss.probabilidades, textos, semantica, previsao.grupos,
                                    robusto.CONFIG["menor_grupo_gaussiano"])
    sub = pd.DataFrame({"id": ids, "model": modelo.gaussiano.classes[rotulos]})
    p = g.softmax(L, axis=1)
    previsao = g.Resultado(sub, p, previsao.grupos, previsao.diagnostico | {
        "atribuicao": f"hungaro por grupo inferido (corte {atribuicao.CORTE}, teto {atribuicao.TETO}), Gaussian 70% + base 30%"})
    if sub.columns.tolist() != ["id", "model"] or sub.id.tolist() != ids.tolist() or not sub.id.is_unique:
        raise ValueError("IDs/ordem/colunas da submissão inconsistentes")
    if len(sub) != len(ids) or not set(sub.model) <= set(modelo.gaussiano.classes):
        raise ValueError("Submissão incompleta ou classes inesperadas")
    if p.shape != (len(ids), 18) or not np.isfinite(p).all() or (p < 0).any() or not np.allclose(p.sum(1), 1):
        raise ValueError("Probabilidades inválidas")
    tmp = saida / "submission.tmp.parquet"
    sub.to_parquet(tmp, index=False)
    tmp.replace(saida / "submission.parquet")
    reaberto = pd.read_parquet(saida / "submission.parquet")
    pd.testing.assert_frame_equal(sub, reaberto)
    np.savez_compressed(saida / "probabilidades.npz", ids=ids, classes=modelo.gaussiano.classes,
                        probabilidades=p, grupos=previsao.grupos)
    _, inv, counts = np.unique(previsao.grupos, return_inverse=True, return_counts=True)
    pd.DataFrame({"id": ids, "grupo_inferido": previsao.grupos, "tamanho_grupo": counts[inv],
                  "componente": np.where(counts[inv] >= 4, "gaussiano5", "mean5")}).to_csv(saida / "roteamento.csv", index=False)
    log.update({"concluido": True, "segundos_total": time.monotonic() - inicio,
                "classes": modelo.gaussiano.classes.tolist(), "roteamento": previsao.diagnostico,
                "grupos_por_tamanho": {str(k): int((counts == k).sum()) for k in np.unique(counts)},
                "distribuicao_predita": sub.model.value_counts().to_dict(),
                "probas_finitas": True, "ids_completos_preservados": True,
                "submission_sha256": sha_arquivo(saida / "submission.parquet")})
    salvar_json(saida / "diagnostico.json", log)
    print("Final:", json.dumps({k: log[k] for k in ("linhas", "ids_sha256", "roteamento", "distribuicao_predita", "segundos_total")}), flush=True)
    return previsao


In [ ]:
try:
    resultado = executar()
except Exception as exc:
    print("FALHA EXPLÍCITA:", type(exc).__name__, str(exc), flush=True)
    traceback.print_exc()
    raise
